# gb-grid: explore the database

Quick visual sanity-check of the Postgres tables populated by `gb-grid`.
Run `gb-grid backfill --from YYYY-MM-DD --to YYYY-MM-DD` first to have data to plot.

Connects via `GB_GRID_DATABASE_URL` (set automatically inside `nix develop`).

**Kernel:** select `Python (gb-grid)` in VS Code (top-right kernel picker) or Jupyter. If it isn't listed, run `just kernel` (or `./scripts/register-kernel.sh`) once.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

_SRC = (Path.cwd() / '..' / 'src').resolve()
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from gb_grid.db import connect
from gb_grid.config import database_url

con = connect()
print('db:', database_url())

def q(sql, params=()):
    with con.cursor() as cur:
        cur.execute(sql, params)
        cols = [d.name for d in cur.description] if cur.description else []
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

q("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""")

## Row counts and latest timestamps

In [ ]:
q("""
    SELECT 'fuelinst' AS t, count(*) AS rows, max(publish_time)::TEXT AS latest FROM fuelinst
    UNION ALL SELECT 'b1610', count(*), max(settlement_date)::TEXT FROM b1610
    UNION ALL SELECT 'boalf', count(*), max(time_from)::TEXT FROM boalf
    UNION ALL SELECT 'system_prices', count(*), max(settlement_date)::TEXT FROM system_prices
""")

## Generation mix over time (FUELINST)

In [ ]:
mix = q("""
    SELECT publish_time, fuel_type, generation_mw
    FROM fuelinst
    WHERE publish_time >= (SELECT max(publish_time) - INTERVAL '7 days' FROM fuelinst)
    ORDER BY publish_time
""")

if mix.empty:
    print('No FUELINST data yet — run a backfill first.')
else:
    pivot = mix.pivot_table(index='publish_time', columns='fuel_type', values='generation_mw', aggfunc='mean').fillna(0)
    ax = pivot.plot.area(figsize=(12, 5), linewidth=0)
    ax.set_ylabel('MW')
    ax.set_title('GB generation mix (last 7 days of available data)')
    ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    plt.tight_layout()
    plt.show()

## Average MW by fuel type

In [ ]:
avg = q("""
    SELECT fuel_type, avg(generation_mw) AS avg_mw
    FROM fuelinst
    WHERE publish_time >= (SELECT max(publish_time) - INTERVAL '7 days' FROM fuelinst)
    GROUP BY 1 ORDER BY 2 DESC
""")

if not avg.empty:
    ax = avg.plot.bar(x='fuel_type', y='avg_mw', figsize=(10, 4), legend=False)
    ax.set_ylabel('avg MW')
    ax.set_title('Average generation by fuel type (last 7 days)')
    plt.tight_layout()
    plt.show()
avg

## System imbalance prices

In [ ]:
prices = q("""
    SELECT settlement_date, settlement_period,
           system_sell_price, system_buy_price, net_imbalance_volume
    FROM system_prices
    ORDER BY settlement_date, settlement_period
""")

if prices.empty:
    print('No system_prices data yet.')
else:
    prices['ts'] = pd.to_datetime(prices['settlement_date']) + pd.to_timedelta((prices['settlement_period'] - 1) * 30, unit='m')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax1.plot(prices['ts'], prices['system_buy_price'], label='buy', color='C3')
    ax1.plot(prices['ts'], prices['system_sell_price'], label='sell', color='C0')
    ax1.set_ylabel('£/MWh'); ax1.legend(); ax1.set_title('System imbalance prices')
    ax2.plot(prices['ts'], prices['net_imbalance_volume'], color='C2')
    ax2.axhline(0, color='k', linewidth=0.5)
    ax2.set_ylabel('NIV (MWh)'); ax2.set_title('Net imbalance volume')
    plt.tight_layout()
    plt.show()

## Top BM units by output (B1610)

In [ ]:
top = q("""
    SELECT bm_unit, sum(quantity_mw) AS total_mwh_proxy
    FROM b1610
    GROUP BY 1 ORDER BY 2 DESC LIMIT 20
""")

if not top.empty:
    ax = top.plot.barh(x='bm_unit', y='total_mwh_proxy', figsize=(8, 6), legend=False)
    ax.invert_yaxis()
    ax.set_title('Top 20 BM units by summed half-hourly output')
    plt.tight_layout()
    plt.show()
top

## Per-BMU dispatch: PN, SO turnup, BOA & SO curtailment

Pick a date range and a list of NGC BMU names (e.g. `PEHE-1`, `MOWWO-1`). Requires `pn`, `boalf` (and optionally `mels`) to be populated for that range.

In [ ]:
from datetime import datetime

from gb_grid.analytics import bmu_dispatch

BMUS = ['PEHE-1', 'MOWWO-1', 'BLHLB-4', 'SVRP-10', 'SGRWO-4', 'KILSB-1']
START = datetime(2026, 4, 1, 0, 0)
END   = datetime(2026, 4, 7, 0, 0)

df = bmu_dispatch(con, BMUS, START, END, freq='1min')
print('rows:', len(df), 'units found:', sorted(df['ngc_bm_unit'].unique()))
df.head()

In [ ]:
from gb_grid.analytics import fetch_b1610

b1610 = fetch_b1610(con, BMUS, START, END)

units = [u for u in BMUS if u in df['ngc_bm_unit'].values]
fig, axes = plt.subplots(len(units), 1, figsize=(12, 3 * max(len(units), 1)), sharex=True)
if len(units) == 1:
    axes = [axes]

for ax, unit in zip(axes, units):
    sub = df[df['ngc_bm_unit'] == unit].set_index('ts')
    pn = sub['pn_mw']
    dispatched = sub['boa_level_mw']
    so_curt = sub['so_curtailment_mw']
    mel = sub['mel_mw']

    # PN baseline + dispatched (post-BOA) line.
    ax.plot(sub.index, pn, color='C0', linewidth=1.4, label='PN (planned)', zorder=2)
    ax.plot(sub.index, dispatched, color='dimgray', linewidth=0.9, label='Dispatched', zorder=2)
    # MEL physical cap.
    ax.plot(sub.index, mel, color='C4', linestyle='--', linewidth=0.8, label='MEL', zorder=3)

    # Turn-up: green band ABOVE PN where dispatched > PN.
    ax.fill_between(
        sub.index, pn, dispatched, where=(dispatched > pn),
        color='C2', alpha=0.35, linewidth=0, label='SO turn-up', zorder=1,
    )
    # BOA curtailment: red band BELOW PN where dispatched < PN.
    ax.fill_between(
        sub.index, dispatched, pn, where=(dispatched < pn),
        color='C3', alpha=0.30, linewidth=0, label='BOA curtailment', zorder=1,
    )
    # SO curtailment overlay (subset of BOA curtailment).
    ax.fill_between(
        sub.index, pn - so_curt, pn, where=(so_curt > 0),
        color='C1', alpha=0.45, linewidth=0, label='SO curtailment', zorder=1,
    )

    # B1610 actual generation on top, in solid black.
    actual = b1610[b1610['ngc_bm_unit'] == unit]
    if not actual.empty:
        ax.plot(actual['ts'], actual['quantity_mw'],
                color='black', linewidth=3, drawstyle='steps-post',
                label='B1610 actual', zorder=5)

    ax.axhline(0, color='k', linewidth=0.4)
    ax.set_ylabel('MW')
    ax.set_title(unit)
    ax.legend(loc='upper right', fontsize=8)

fig.suptitle(f'BMU dispatch  {START:%Y-%m-%d}  –  {END:%Y-%m-%d}')
plt.tight_layout()
plt.show()

In [ ]:
# Daily totals (MWh ≈ MW averaged * hours)
hours = (END - START).total_seconds() / 3600
totals = (
    df.groupby('ngc_bm_unit')[['so_turnup_mw', 'boa_curtailment_mw', 'so_curtailment_mw']]
      .mean()
      .mul(hours)
      .rename(columns=lambda c: c.replace('_mw', '_mwh'))
      .round(1)
)
totals